# ts_detrend

> 对应代码：`EconometricsCode/code_in_notes/Chap.29/ts_detrend.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.29`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.29")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入季度 GDP 数据，声明时间变量，并取对数得到 ln_gdp 后先画出其时间序列图：对数 GDP 呈现明显的上升趋势，需要先去除趋势才能研究其平稳成分。

In [ ]:
%%stata
use ../datasets/quarterlyGDP.dta, clear
tsset time
gen ln_gdp = log(qgdp)
label variable ln_gdp "log(GDP)"
tsline ln_gdp

构造时间下标 t，并将其中心化、缩小为 td = (t − (N+1)/2)/10。中心化使一次项与二次项近似正交、降低共线性，除以 10 则把系数放大到可读的量级；再生成平方项 t2。

In [ ]:
%%stata
gen t  = _n
gen td = (t - (_N+1)/2)/10
gen t2 = td^2

将 ln_gdp 对时间的一次项和二次项做 OLS 回归，用二次多项式拟合确定性趋势；取残差作为去趋势后的序列并画图。注意这里回归实际用的是 t 与 t2（未中心化的 t 与中心化后再平方的 t2 组合），残差刻画了偏离平滑趋势的波动成分。

In [ ]:
%%stata
reg ln_gdp t t2
predict detrend_gdp, resid
label variable detrend_gdp "去趋势"
tsline detrend_gdp

在趋势回归中加入季度虚拟变量 i.quarter，把确定性的季节成分也吸收掉，再取残差并画图。比较两图可以看到，季节调整后的去趋势序列波动更规则，更接近我们想要的平稳成分。

In [ ]:
%%stata
reg ln_gdp t t2 i.quarter
predict detrend_quarter_gdp, resid
label variable detrend_quarter_gdp "去趋势及季节项"
tsline detrend_quarter_gdp

画出同时去除趋势与季节项后序列的自相关函数。若残差 ACF 仍然衰减缓慢，说明确定性的多项式趋势并不足以使序列平稳——这正是需要用单位根检验与差分方法（见 ts_detrend_diff.do）的动机。

In [ ]:
%%stata
ac detrend_quarter_gdp